# 06 - Cliff's Delta Non-Parametric Effect Sizes
Calculate pairwise Cliff's delta values for each metric and language pair.

In [ ]:
from itertools import combinations
import os
import os
import pandas as pd

# Run from this notebook's folder. The bundled CSV is ../data/.
data_path = os.path.join("..", "data", "final_analysis_dataset.csv")
if not os.path.exists(data_path):
    data_path = "/mnt/data/final_analysis_dataset.csv"
df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
metrics = ['TCR', 'TCI', 'TMR']
def cliffs_delta(a, b):
    a, b = list(a), list(b)
    more = sum(x > y for x in a for y in b)
    less = sum(x < y for x in a for y in b)
    d = (more - less) / (len(a) * len(b))
    ad = abs(d)
    mag = 'negligible' if ad < .147 else 'small' if ad < .33 else 'medium' if ad < .474 else 'large'
    return d, mag
languages = df['Language'].dropna().unique()
rows = []
for m in metrics:
    for l1, l2 in combinations(languages, 2):
        a = df.loc[df['Language'] == l1, m].dropna().values
        b = df.loc[df['Language'] == l2, m].dropna().values
        d, mag = cliffs_delta(a, b)
        rows.append({'Metric':m,'Comparison':f'{l1} vs {l2}','Cliffs_Delta':round(d,4),'Magnitude':mag})
cliffs_df = pd.DataFrame(rows)
os.makedirs("../outputs/tables", exist_ok=True)
cliffs_df.to_csv("../outputs/tables/cliffs_delta_effect_sizes.csv", index=False)
cliffs_df